# MedSafe — Drug Interaction Exploration Notebook

This notebook walks through the MedSafe pipeline interactively:
1. Load the processed DDI graph and inspect its structure
2. Load trained GIN embeddings and visualize the drug embedding space
3. Run pairwise interaction predictions with the R-GCN model
4. Compute and visualize polypharmacy risk scores
5. Generate explanations with GNNExplainer and Shapley values

> **Prerequisites**: Run `python train.py` before using this notebook.

In [ ]:
import sys
sys.path.insert(0, '..')  # Add medsafe root to path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from configs.loader import load_config
cfg = load_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Config loaded: GIN hidden_dim={cfg.gin.hidden_dim}, RGCN num_layers={cfg.rgcn.num_layers}')

## 1. Load the DDI Graph

In [ ]:
from pathlib import Path

graph_path = Path('../data/graphs/ddi_hetero_graph.pt')
assert graph_path.exists(), 'DDI graph not found. Run: python pipeline/run_pipeline.py'

ddi_graph = torch.load(graph_path, map_location='cpu', weights_only=False)

print('=== DDI Graph Summary ===')
print(f'Drug nodes:   {ddi_graph["drug"].num_nodes:,}')
print(f'Target nodes: {ddi_graph["target"].num_nodes:,}')

for edge_type in ddi_graph.edge_types:
    ei = ddi_graph[edge_type].edge_index
    print(f'Edge type {edge_type}: {ei.shape[1]:,} edges')

drug_names = getattr(ddi_graph, 'drug_names', [])
print(f'\nFirst 10 drugs: {drug_names[:10]}')

## 2. Load Drug Embeddings & Visualize (t-SNE)

In [ ]:
from pathlib import Path

emb_path = Path('../data/embeddings/drug_embeddings.pt')
ids_path  = Path('../data/embeddings/embedding_drug_ids.parquet')

assert emb_path.exists(), 'Embeddings not found. Run: python train.py'

embeddings = torch.load(emb_path, map_location='cpu', weights_only=False)
drug_ids_df = pd.read_parquet(ids_path)

print(f'Embedding shape: {embeddings.shape}')
print(f'Drug IDs: {len(drug_ids_df):,}')

# t-SNE visualization (sample 1000 for speed)
from sklearn.manifold import TSNE
import torch.nn.functional as F

n_sample = min(1000, len(drug_ids_df))
sample_idx = np.random.choice(len(drug_ids_df), n_sample, replace=False)
emb_norm = F.normalize(embeddings[sample_idx], dim=-1).numpy()

print('Running t-SNE...')
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
emb_2d = tsne.fit_transform(emb_norm)

plt.figure(figsize=(12, 9))
plt.scatter(emb_2d[:, 0], emb_2d[:, 1], alpha=0.5, s=10, c='#6366f1')
plt.title('GIN Drug Embedding Space (t-SNE, n=1000)', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.savefig('../evaluation/results/tsne_notebook.png', dpi=150)
plt.show()

## 3. Load R-GCN Model & Run Predictions

In [ ]:
from models.rgcn_predictor import build_rgcn_predictor
from training.finetune_rgcn import build_combined_edge_index

ckpt_path = Path('../checkpoints/rgcn_finetune/rgcn_best.pt')
assert ckpt_path.exists(), 'Model checkpoint not found. Run: python train.py'

drug_x = ddi_graph['drug'].x
model = build_rgcn_predictor(cfg, drug_x.shape[1])

ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()

print(f'Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters')

edge_index, edge_type = build_combined_edge_index(ddi_graph, torch.device('cpu'))
print(f'Graph: {edge_index.shape[1]:,} edges, {edge_type.max().item()+1} edge types')

In [ ]:
# Predict a specific pair
drug_to_idx = getattr(ddi_graph, 'drug_to_idx', {})
idx_to_drug = {v: k for k, v in drug_to_idx.items()}

def predict_pair(drug_a_name, drug_b_name):
    idx_a = drug_to_idx.get(drug_a_name, -1)
    idx_b = drug_to_idx.get(drug_b_name, -1)
    if idx_a < 0 or idx_b < 0:
        print(f'Drug not found: {drug_a_name if idx_a < 0 else drug_b_name}')
        return

    src = torch.tensor([idx_a])
    dst = torch.tensor([idx_b])

    with torch.no_grad():
        binary, severity, itype, faers = model(drug_x, edge_index, edge_type, src, dst)

    prob = torch.sigmoid(binary).item()
    sev_probs = torch.softmax(severity, dim=-1).squeeze().tolist()
    pred_sev = int(torch.argmax(severity).item())

    sev_labels = ['Minor', 'Moderate', 'Major', 'Contraindicated']
    print(f'\n{drug_a_name} + {drug_b_name}')
    print(f'  Interaction probability: {prob:.1%}')
    print(f'  Predicted severity:      {sev_labels[pred_sev]} ({pred_sev}/3)')
    print(f'  Severity distribution:   {[f"{p:.1%}" for p in sev_probs]}')

# Test known high-risk pairs
predict_pair('DB00682', 'DB00945')  # Warfarin + Aspirin (use DrugBank IDs)
predict_pair('DB00682', 'DB00331')  # Warfarin + Metformin (should be lower risk)

## 4. Polypharmacy Risk Score

In [ ]:
from scoring.polypharmacy_score import compute_polypharmacy_score

# Test with a high-risk combination
drug_list = ['Warfarin', 'Aspirin', 'Ibuprofen', 'Metformin', 'Lisinopril']
report = compute_polypharmacy_score(drug_list, include_shapley=True)

print(f'\n=== Polypharmacy Report ===')
print(f'Risk Score: {report.overall_risk_score:.1f}/100 [{report.risk_tier.upper()}]')
print(f'Summary:    {report.summary}')
print(f'\nSpecial Flags ({len(report.special_flags)}):')
for f in report.special_flags:
    print(f'  [{f.severity.upper()}] {f.flag_type}: {f.drugs_involved}')

print(f'\nFlagged Interactions ({report.num_flagged}):')
for pair in report.flagged_interactions[:3]:
    print(f'  {pair.drug_a} + {pair.drug_b}: {pair.severity_label} ({pair.confidence:.0%} confidence)')

print(f'\nShapley Values (risk contribution):')
for drug, sv in sorted(report.shapley_values.items(), key=lambda x: x[1], reverse=True):
    print(f'  {drug:<20} {sv:.2f}')

## 5. Interaction Heatmap

In [ ]:
import seaborn as sns

drugs = drug_list
matrix = report.interaction_matrix

# Build numpy matrix
n = len(drugs)
heat = np.zeros((n, n))
for i, a in enumerate(drugs):
    for j, b in enumerate(drugs):
        if a != b:
            heat[i, j] = matrix.get(a, {}).get(b, 0)

fig, ax = plt.subplots(figsize=(8, 6))
cmap = mcolors.LinearSegmentedColormap.from_list(
    'risk', ['#1e293b', '#f59e0b', '#f97316', '#ef4444']
)
sns.heatmap(
    heat, ax=ax,
    xticklabels=drugs, yticklabels=drugs,
    cmap=cmap, vmin=0, vmax=3,
    linewidths=0.5, linecolor='#334155',
    annot=True, fmt='.0f',
    cbar_kws={'label': 'Severity (0=minor, 3=contraindicated)'}
)
ax.set_title('Drug Interaction Severity Matrix', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('../evaluation/results/interaction_heatmap.png', dpi=150)
plt.show()

## 6. MC Dropout Uncertainty

In [ ]:
from explainability.monte_carlo_dropout import mc_dropout_predict

# Predict Warfarin + Aspirin with uncertainty
# (replace with actual node indices from your graph)
warfarin_idx = drug_to_idx.get('DB00682', 0)
aspirin_idx  = drug_to_idx.get('DB00945', 1)

uc = mc_dropout_predict(
    model=model,
    drug_x=drug_x,
    edge_index=edge_index,
    edge_type=edge_type,
    drug_i_idx=warfarin_idx,
    drug_j_idx=aspirin_idx,
    n_samples=50,
    support_count=100,
)

print(f'Interaction probability: {uc.mean_prob:.1%} ± {uc.std_prob:.1%}')
print(f'Confidence level: {uc.confidence_level} ({uc.confidence_score:.1%})')
print(f'Severity distribution: {[f"{p:.1%}" for p in uc.severity_mean]}')
if uc.warning_message:
    print(f'Warning: {uc.warning_message}')